# 12 · Matrix factorizations / Factorizaciones matriciales

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/12-matrix-factorizations.ipynb)

*deep dive · take-home / estudio a fondo · para después*

The workshop used three matrix factorizations without ever naming the family: **SVD** behind the pseudoinverse in section 07 and behind deconvolution in section 09, and **Cholesky** in one of the take-homes in section 11.

This deep dive puts the whole family in one place — **LU, QR, Cholesky, eigendecomposition, SVD and NMF** — and asks the same question of each: *what does this factorization make easy that was hard before?*

> 🇪🇸 El taller usó tres factorizaciones matriciales sin nombrar nunca a la familia: la **SVD** detrás de la pseudoinversa en la sección 07 y de la deconvolución en la 09, y **Cholesky** en uno de los ejercicios para casa de la sección 11.
>
> Este estudio a fondo reúne a toda la familia — **LU, QR, Cholesky, descomposición espectral, SVD y NMF** — y le hace a cada una la misma pregunta: *¿qué vuelve fácil esta factorización que antes era difícil?*

## What you will be able to do / Lo que podrás hacer

- Say which problem each of **LU, QR, Cholesky, eigendecomposition, SVD and NMF** is the right tool for.
- Explain what each factorization assumes about its input, and what it does when that assumption fails.
- Solve the same real least-squares problem three ways and compare cost against numerical stability.
- Connect the SVD you already used in sections 07 and 09 to the rest of the family.
- Read a factorization's factors as structure in the data rather than as an opaque numerical result.

> 🇪🇸
>
> - Decir para qué problema es la herramienta adecuada cada una de **LU, QR, Cholesky, descomposición espectral, SVD y NMF**.
> - Explicar qué supone cada factorización sobre su entrada y qué ocurre cuando ese supuesto no se cumple.
> - Resolver el mismo problema real de mínimos cuadrados de tres formas y comparar coste frente a estabilidad numérica.
> - Conectar la SVD que ya usaste en las secciones 07 y 09 con el resto de la familia.
> - Leer los factores de una factorización como estructura de los datos y no como un resultado numérico opaco.

## Start with an everyday analogy / Empecemos con una analogía cotidiana

Think about how you factor a number.

`3960 = 2³ × 3² × 5 × 11`

Nothing was added and nothing was lost. The same number is written a second way — and in that second form, questions that were hard become easy. *Is it divisible by 9?* Read it off. *What is the largest square that divides it?* Read it off.

A **matrix factorization** does exactly this for a matrix. `A` becomes a product of two or three matrices with useful shapes — triangular, orthogonal, diagonal, nonnegative — and questions that were expensive on `A` become cheap on the factors.

So the question this deep dive keeps asking is never *"how does the algorithm work?"* It is:

> **Which question does this factorization make easy, and what did it cost me to get there?**

> 🇪🇸 Piensa en cómo se factoriza un número: `3960 = 2³ × 3² × 5 × 11`. No se añade ni se pierde nada, pero en esa segunda forma las preguntas difíciles se vuelven fáciles.
>
> Una **factorización matricial** hace exactamente eso con una matriz: `A` se convierte en un producto de matrices con formas útiles — triangular, ortogonal, diagonal, no negativa — y las preguntas caras sobre `A` se vuelven baratas sobre los factores.
>
> La pregunta de este estudio a fondo nunca es *"¿cómo funciona el algoritmo?"*, sino:
>
> **¿Qué pregunta vuelve fácil esta factorización y cuánto me costó llegar ahí?**

## Six factorizations, six questions / Seis factorizaciones, seis preguntas

| Factorization | Answers | Requires |
|---|---|---|
| **LU** | Solve `Ax = b` repeatedly for the same square `A`. | `A` square |
| **Cholesky** | The same, when `A` is symmetric positive definite — at half the cost. | `A` symmetric, positive definite |
| **QR** | Least squares, without ever forming `XᵀX`. | nothing |
| **Eigendecomposition** | What does repeated application of `A` converge to? | `A` square (best behaved when symmetric) |
| **SVD** | The best rank-`k` approximation of *any* matrix. | nothing |
| **NMF** | Parts you can name, when negative numbers have no meaning. | `A ≥ 0` |

Two of these you have already used without being told their family name. The **SVD** was behind the pseudoinverse in section 07 and behind deconvolution in section 09; **Cholesky** built correlated returns in one of section 11's take-homes. This notebook puts all six side by side and makes each one answer for its cost.

> 🇪🇸 Seis factorizaciones, una pregunta para cada una: **LU** para resolver `Ax = b` muchas veces con la misma `A`; **Cholesky** para lo mismo cuando `A` es simétrica definida positiva, a mitad de coste; **QR** para mínimos cuadrados sin formar nunca `XᵀX`; la **descomposición espectral** para saber a qué converge la aplicación repetida de `A`; la **SVD** para la mejor aproximación de rango `k` de *cualquier* matriz; y **NMF** para estructura por partes cuando los números negativos no significan nada.
>
> Ya usaste dos de ellas sin conocer el nombre de la familia: la **SVD** detrás de la pseudoinversa (sección 07) y de la deconvolución (sección 09), y **Cholesky** en uno de los ejercicios para casa de la sección 11.

## Setup / Preparación

Run this cell first. Everything below depends on it.

It loads four things the workshop already uses, so there is nothing new to download except the airline CSV you also fetched in section 08:

1. **Two real images** — `astronaut()` and `camera()` from `skimage.data`, both 512×512 in grayscale.
2. **Real handwritten digits** — `load_digits()`, 1797 images of 8×8 pixels, flattened to a 1797×64 matrix.
3. **The real monthly airline series** from section 08 — 144 months of passenger counts.
4. **A flop-count table**, used later to turn big-O into predicted seconds.

> 🇪🇸 Ejecuta primero esta celda; todo lo demás depende de ella.
>
> Carga cuatro cosas que el taller ya usa, así que no hay nada nuevo que descargar salvo el CSV de aerolíneas que también obtuviste en la sección 08:
>
> 1. **Dos imágenes reales** — `astronaut()` y `camera()`, ambas de 512×512 en escala de grises.
> 2. **Dígitos manuscritos reales** — 1797 imágenes de 8×8 píxeles, aplanadas a una matriz de 1797×64.
> 3. **La serie real mensual de aerolíneas** de la sección 08 — 144 meses.
> 4. **Una tabla de conteo de flops**, que después convierte la notación big-O en segundos predichos.

In [ ]:
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets

from IPython.display import display
from scipy.linalg import cho_factor, cho_solve, lu_factor
from skimage import data
from skimage.color import rgb2gray
from sklearn.datasets import load_digits
from sklearn.decomposition import NMF

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

FLIGHTS = (
    "https://raw.githubusercontent.com/mwaskom/"
    "seaborn-data/master/flights.csv"
)

# 1. Two real images, both already workshop datasets.
IMAGES = {
    "astronaut": rgb2gray(data.astronaut()),
    "camera": data.camera().astype(float) / 255.0,
}

# 2. Real handwritten digits, scaled to [0, 1]: 1797 rows, 64 pixel columns.
digits = load_digits()
D = digits.data / 16.0
digit_labels = digits.target

# 3. The real monthly airline series from section 08.
flights = pd.read_csv(FLIGHTS)
passengers = flights["passengers"].to_numpy(float)
month = np.arange(len(passengers), dtype=float)
month_scaled = month / month.max()          # rescaled to [0, 1]

rng = np.random.default_rng(0)

# 4. Flop counts for a dense m x n factorization with m >= n, from
# Trefethen & Bau, *Numerical Linear Algebra*. Only the randomized SVD
# uses the target rank k.
FLOPS = {
    "Cholesky": lambda m, n, k: n ** 3 / 3,
    "LU": lambda m, n, k: 2 * n ** 3 / 3,
    "QR": lambda m, n, k: 2 * m * n ** 2 - 2 * n ** 3 / 3,
    "Eigendecomposition": lambda m, n, k: 9 * n ** 3,
    "Thin SVD": lambda m, n, k: 2 * m * n ** 2 + 11 * n ** 3,
    "Randomized SVD": lambda m, n, k: 4 * m * n * k,
}

def relative_error(reference, approximation):
    return (np.linalg.norm(reference - approximation)
            / np.linalg.norm(reference))

def psnr(reference, approximation, peak=1.0):
    """Peak signal-to-noise ratio in dB, for images scaled to [0, peak]."""
    mse = np.mean((reference - approximation) ** 2)
    return 10 * np.log10(peak ** 2 / mse)

def best_time(call, repeats=3):
    """Fastest of `repeats` runs — the least noisy estimator of a timing."""
    times = []
    for _ in range(repeats):
        start = time.perf_counter()
        call()
        times.append(time.perf_counter() - start)
    return min(times)

print("Images / Imágenes:", {k: v.shape for k, v in IMAGES.items()})
print("Digit matrix / Matriz de dígitos:", D.shape)
print("Airline months / Meses de aerolíneas:", passengers.shape)
print("Flop formulas / Fórmulas de flops:", len(FLOPS))
print()
print("EN: Setup ready.")
print("ES: Preparación lista.")

## Why this matters / Por qué esto importa

Every factorization here is available as a one-line call. `np.linalg.svd`, `np.linalg.qr`, `np.linalg.cholesky` — none of them is hard to *type*. That is exactly the problem.

The cost of picking the wrong one is invisible at the call site and expensive everywhere else:

- Forming `XᵀX` to solve least squares squares the condition number. On the airline fit below, that turns a coefficient error of `2e-14` into `2e-03` — **eleven orders of magnitude**, from one line of algebra that looked harmless.
- Calling `np.linalg.inv(A) @ B` instead of factoring once and solving is both slower *and* less accurate. You will measure both.
- Reaching for a full SVD when you need 20 components out of 1682 does `O(mn·min(m,n))` work to keep `O(mn)` of it.

### Learning cycle / Ciclo de aprendizaje

Before each exercise:

**Predict → Run → Explain / Predice → Ejecuta → Explica**

Ask:

1. What shape is this matrix — tall, square, symmetric, nonnegative?
2. Am I solving once, or many times with the same matrix?
3. Do I need all of the factorization, or only the top `k`?
4. Do I need factors a human can read, or only a small reconstruction error?

> 🇪🇸 Todas estas factorizaciones son una sola línea de código. Ese es justo el problema: el coste de elegir mal es invisible donde la llamas y caro en todas partes.
>
> Formar `XᵀX` eleva al cuadrado el número de condición: en el ajuste de aerolíneas de abajo, convierte un error de `2e-14` en uno de `2e-03`. Llamar a `np.linalg.inv(A) @ B` es más lento *y* menos preciso que factorizar una vez. Y pedir una SVD completa cuando solo necesitas 20 componentes hace trabajo `O(mn·min(m,n))` para conservar `O(mn)`.
>
> Antes de cada ejercicio pregunta:
>
> 1. ¿Qué forma tiene la matriz: alta, cuadrada, simétrica, no negativa?
> 2. ¿Resuelvo una vez o muchas con la misma matriz?
> 3. ¿Necesito toda la factorización o solo las primeras `k` componentes?
> 4. ¿Necesito factores legibles o solo un error de reconstrucción pequeño?

## 12.1 Every factorization is a constrained optimization / Toda factorización es una optimización con restricciones

The six look like six unrelated algorithms. They are not. Each one is the answer to *minimize this, subject to that* — and the constraint is what gives the factors their shape.

| Factorization | Minimize / maximize | Subject to |
|---|---|---|
| **QR** | `‖y − Xβ‖₂` | `X = QR`, `Q` orthonormal, `R` upper-triangular |
| **Truncated SVD** | `‖A − B‖_F` | `rank(B) ≤ k` |
| **Eigendecomposition** | maximize `vᵀAv` | `‖v‖ = 1`, `A` symmetric |
| **NMF** | `‖V − WH‖_F²` | `W ≥ 0`, `H ≥ 0` |
| **Cholesky** | (feasibility, not optimality) | `A = LLᵀ`, `L` lower-triangular, positive diagonal |
| **LU** | (feasibility, not optimality) | `PA = LU`, `L` unit lower-, `U` upper-triangular |

Three consequences follow directly, and all three are things you can check numerically rather than take on faith:

1. **The truncated SVD is optimal, and provably so.** The Eckart–Young–Mirsky theorem says no rank-`k` matrix of any kind gets closer to `A` in Frobenius norm, and it tells you the error exactly: `‖A − A_k‖_F = √(Σ_{i>k} σᵢ²)`. You will verify that equality to machine precision in §12.4.
2. **NMF cannot beat it on error, and is not trying to.** Adding `W, H ≥ 0` shrinks the feasible set, so the best achievable error goes *up*. What you buy with that error is factors you can look at and name. §12.6 measures both sides of that trade.
3. **Cholesky and LU are not optimizing anything.** They are exact rewrites of `A`, and their value is entirely downstream: once you hold the factors, a solve costs `O(n²)` instead of `O(n³)`.

> 🇪🇸 Las seis parecen seis algoritmos sin relación. No lo son: cada una es la respuesta a *minimiza esto, sujeto a aquello*, y la restricción es lo que da forma a los factores.
>
> De ahí se siguen tres consecuencias, y las tres se pueden comprobar numéricamente:
>
> 1. **La SVD truncada es óptima**, y el teorema de Eckart–Young–Mirsky da el error exacto: `‖A − A_k‖_F = √(Σ_{i>k} σᵢ²)`. Lo verificarás en §12.4.
> 2. **NMF no puede superarla en error** — añadir `W, H ≥ 0` reduce el conjunto factible — y no lo intenta: lo que compras con ese error son factores legibles. §12.6 mide ambos lados.
> 3. **Cholesky y LU no optimizan nada.** Son reescrituras exactas de `A`, y su valor está río abajo: con los factores en la mano, resolver cuesta `O(n²)` en vez de `O(n³)`.

### Interactive method chooser / Selector interactivo de método

Tick the properties your data actually has and read back which factorization fits, why, and what it costs.

The point is not to memorize the table. It is that **four or five yes/no facts about your matrix determine the answer**, and you almost always know those facts before you write any code.

> 🇪🇸 Marca las propiedades que realmente tienen tus datos y lee qué factorización encaja, por qué y cuánto cuesta.
>
> Lo importante no es memorizar la tabla, sino que **cuatro o cinco hechos de sí/no sobre tu matriz determinan la respuesta**, y casi siempre los conoces antes de escribir código.

In [ ]:
square = widgets.Checkbox(value=False, description="Square / Cuadrada")
symmetric = widgets.Checkbox(value=False, description="Symmetric / Simétrica")
posdef = widgets.Checkbox(
    value=False, description="Positive definite / Definida positiva")
sparse = widgets.Checkbox(value=False, description="Sparse / Dispersa")
nonneg = widgets.Checkbox(
    value=False, description="Nonnegative / No negativa")
readable = widgets.Checkbox(
    value=False, description="Factors must be readable / Factores legibles")
low_rank = widgets.Checkbox(
    value=False, description="Only top k needed / Solo las primeras k")
many_rhs = widgets.Checkbox(
    value=False, description="Many right-hand sides / Muchos lados derechos")

def choose(square, symmetric, posdef, sparse, nonneg,
           readable, low_rank, many_rhs):
    # Ordered most-specific first: the first rule that fires wins, which is
    # how a practitioner actually decides.
    if nonneg and readable:
        pick = "NMF"
        why_en = ("nonnegative data plus a demand for readable factors is the "
                  "one case where giving up optimal error is the right trade.")
        why_es = ("datos no negativos más la exigencia de factores legibles: "
                  "el único caso donde renunciar al error óptimo compensa.")
        cost = "O(mnk) per iteration / por iteración"
    elif low_rank and sparse:
        pick = "Lanczos / truncated SVD (scipy.sparse.linalg.svds)"
        why_en = ("a sparse matrix should never be densified; Lanczos touches "
                  "only the nonzeros.")
        why_es = ("una matriz dispersa nunca debe densificarse; Lanczos toca "
                  "solo los no ceros.")
        cost = "O(k · nnz(A)) per restart / por reinicio"
    elif low_rank:
        pick = "Randomized SVD (sklearn.utils.extmath.randomized_svd)"
        why_en = ("you asked for k components, so do work proportional to k "
                  "rather than to min(m, n).")
        why_es = ("pediste k componentes: haz trabajo proporcional a k y no a "
                  "min(m, n).")
        cost = "O(mnk)"
    elif symmetric and posdef and many_rhs:
        pick = "Cholesky (scipy.linalg.cho_factor / cho_solve)"
        why_en = ("half the flops of LU, and the factorization is reused by "
                  "every right-hand side.")
        why_es = ("la mitad de flops que LU, y la factorización se reutiliza "
                  "en cada lado derecho.")
        cost = "O(n³/3) once, then O(n²) per solve / luego por solución"
    elif symmetric and posdef:
        pick = "Cholesky (np.linalg.cholesky)"
        why_en = "symmetry and positive definiteness halve the work; use them."
        why_es = ("la simetría y la definición positiva reducen el trabajo a "
                  "la mitad; aprovéchalas.")
        cost = "O(n³/3)"
    elif symmetric:
        pick = "Eigendecomposition (np.linalg.eigh)"
        why_en = ("a symmetric matrix has a real orthogonal eigenbasis — eigh "
                  "exploits it, eig does not.")
        why_es = ("una matriz simétrica tiene base propia ortogonal real: eigh "
                  "lo aprovecha, eig no.")
        cost = "O(9n³) with eigenvectors / con autovectores"
    elif square and many_rhs:
        pick = "LU (scipy.linalg.lu_factor / lu_solve)"
        why_en = ("factor once, then every extra solve is a pair of triangular "
                  "substitutions.")
        why_es = ("factoriza una vez y cada solución extra son dos "
                  "sustituciones triangulares.")
        cost = "O(2n³/3) once, then O(n²) per solve / luego por solución"
    elif square:
        pick = "LU (np.linalg.solve)"
        why_en = "np.linalg.solve is LU with pivoting, and never forms an inverse."
        why_es = ("np.linalg.solve es LU con pivoteo y nunca forma una "
                  "inversa.")
        cost = "O(2n³/3)"
    else:
        pick = "QR (np.linalg.qr, or np.linalg.lstsq)"
        why_en = ("a tall matrix means least squares, and QR solves it at "
                  "condition number κ(X) rather than κ(X)².")
        why_es = ("una matriz alta significa mínimos cuadrados, y QR lo "
                  "resuelve con número de condición κ(X) y no κ(X)².")
        cost = "O(2mn² − 2n³/3)"

    print("Use / Usa:", pick)
    print("Cost / Coste:", cost)
    print()
    print("EN:", why_en)
    print("ES:", why_es)

    if posdef and not symmetric:
        print()
        print("EN: note — positive definite is only defined for symmetric "
              "matrices here; tick symmetric too.")
        print("ES: nota — aquí la definición positiva solo tiene sentido para "
              "matrices simétricas; marca también simétrica.")

chooser_output = widgets.interactive_output(
    choose,
    {
        "square": square,
        "symmetric": symmetric,
        "posdef": posdef,
        "sparse": sparse,
        "nonneg": nonneg,
        "readable": readable,
        "low_rank": low_rank,
        "many_rhs": many_rhs,
    },
)

display(widgets.VBox([
    widgets.HBox([square, symmetric, posdef, sparse]),
    widgets.HBox([nonneg, readable, low_rank, many_rhs]),
    chooser_output,
]))

### When the assumption fails / Cuando el supuesto no se cumple

The chooser above reads properties off your matrix. So it is worth knowing what each factorization actually *does* when the property it needs is absent — because the three cases are not alike, and only two of them tell you.

- **Cholesky on a matrix that is not positive definite** raises `LinAlgError`. Loud, immediate, unmissable. The next cell triggers it on the real covariance of 20 digit pixels: one of those pixels never varies — the workshop already found the three constant pixels in section 02 — so the covariance is singular, its smallest eigenvalue is exactly zero, and there is no `L` to find.
- **NMF on data containing negatives** raises `ValueError` before it starts. Also loud.
- **`eigh` on a matrix that is not symmetric returns an answer.** No error, no warning, no hint. `eigh` reads only the lower triangle and mirrors it, so it hands back the eigenvalues of a *different matrix* — a symmetric one you never constructed. This is the dangerous case, and it is the one people hit, because `eigh` is the function you are told to prefer.

That asymmetry is the lesson. A factorization that refuses is doing you a favour; check the assumption yourself before the one that does not.

> 🇪🇸 El selector de arriba lee propiedades de tu matriz, así que conviene saber qué hace cada factorización cuando la propiedad que necesita no está: los tres casos no son iguales y solo dos te avisan.
>
> - **Cholesky sobre una matriz que no es definida positiva** lanza `LinAlgError`. La celda siguiente lo provoca sobre la covarianza real de 20 píxeles de dígitos: uno de esos píxeles nunca varía — el taller ya encontró los tres píxeles constantes en la sección 02 — así que la covarianza es singular y no existe `L`.
> - **NMF sobre datos con negativos** lanza `ValueError` antes de empezar. También ruidoso.
> - **`eigh` sobre una matriz no simétrica devuelve una respuesta.** Sin error ni aviso: `eigh` lee solo el triángulo inferior y lo refleja, así que te da los autovalores de *otra matriz*, una simétrica que nunca construiste. Este es el caso peligroso, y es el que la gente encuentra, porque `eigh` es la función que te recomiendan.
>
> Esa asimetría es la lección: una factorización que se niega te está haciendo un favor. Comprueba tú el supuesto antes de la que no se niega.

In [ ]:
# 1. Cholesky needs positive definiteness, and says so.
pixel_covariance = np.cov(D[:, :20].T)
smallest = np.linalg.eigvalsh(pixel_covariance).min()

print("Covariance of 20 real digit pixels / Covarianza de 20 píxeles reales:",
      pixel_covariance.shape)
print("Constant pixels among them / Píxeles constantes entre ellos:",
      int(np.sum(D[:, :20].var(axis=0) == 0)))
print("Smallest eigenvalue / Menor autovalor:", f"{smallest:.3e}")
print("Rank / Rango:", np.linalg.matrix_rank(pixel_covariance), "of / de 20")
try:
    np.linalg.cholesky(pixel_covariance)
    print("Cholesky succeeded / Cholesky funcionó")
except np.linalg.LinAlgError as exc:
    print("Cholesky raised / Cholesky lanzó:", exc)

# 2. NMF needs nonnegativity, and says so.
try:
    NMF(n_components=2, max_iter=10).fit(D - 0.5)
except ValueError as exc:
    print()
    print("NMF raised / NMF lanzó:", str(exc).split(".")[0])

# 3. eigh needs symmetry, and does NOT say so.
block = D[10:14, 20:24]
eig_values = np.linalg.eig(block)[0]

print()
print("A real 4x4 pixel block / Un bloque real de 4x4 píxeles, symmetric?",
      np.allclose(block, block.T))
# eig returns a complex array for any non-symmetric input. Here the largest
# imaginary part is zero, so dropping it is safe — but check before you do,
# because a non-symmetric matrix is exactly the kind that can have genuinely
# complex eigenvalues (a rotation matrix has a pair of them).
print("Largest imaginary part / Mayor parte imaginaria:",
      f"{np.abs(eig_values.imag).max():.1e}")
print("eig  (correct / correcto) :",
      np.round(np.sort(eig_values.real), 4))
print("eigh (silently wrong / silenciosamente incorrecto):",
      np.round(np.sort(np.linalg.eigh(block)[0]), 4))
print()
print("What eigh actually decomposed / Lo que eigh descompuso de verdad:")
print(np.round(np.tril(block) + np.tril(block, -1).T, 4))
print()
print("EN: eigh mirrored the lower triangle and factorized THAT. Nothing in")
print("    the call signalled it. Check symmetry yourself:")
print("        assert np.allclose(A, A.T)")
print("ES: eigh reflejó el triángulo inferior y factorizó ESO. Nada en la")
print("    llamada lo indicó. Comprueba la simetría tú mismo:")
print("        assert np.allclose(A, A.T)")

## 12.2 The cost table, derived / La tabla de costes, deducida

These are flop counts, not measurements — the leading term of the arithmetic each algorithm performs on a dense `m × n` matrix with `m ≥ n`. They come from Trefethen & Bau, *Numerical Linear Algebra*, and they are already loaded as `FLOPS` in the setup cell.

| Factorization | Flops | Notes |
|---|---|---|
| **Cholesky** (`n × n` SPD) | `n³/3` | half of LU: symmetry means half the matrix is never touched |
| **LU** (`n × n`) | `2n³/3` | with partial pivoting; this is what `np.linalg.solve` runs |
| **QR** (`m × n`) | `2mn² − 2n³/3` | Householder; for `m = n` that is `4n³/3` |
| **Eigendecomposition** (`n × n` symmetric) | `≈ 9n³` | with eigenvectors; `4n³/3` for eigenvalues alone |
| **Thin SVD** (`m × n`) | `2mn² + 11n³` | with `U` and `V`; the most expensive thing here |
| **Randomized SVD** (rank `k`) | `≈ 4mnk` | `k ≪ n`, dense, probabilistic error bound |
| **Lanczos** (rank `k`, sparse) | `O(k · nnz(A))` | touches only the nonzeros, never densifies |

### Three consequences worth stating out loud / Tres consecuencias que conviene decir en voz alta

**1. Factor once, solve many.** A Cholesky factorization costs `n³/3` once. Each subsequent solve is two triangular substitutions at `O(n²)`. So `m` right-hand sides cost

`O(n³ + mn²)`, **not** `O(mn³)`.

At `n = 1797` and `m = 200` — the exact sizes of exercise 2 below — that is the difference between one factorization and two hundred, and the measured gap runs to several hundred times.

**2. The normal equations square the condition number.** Solving least squares by `β = (XᵀX)⁻¹Xᵀy` is one line and it is a trap, because

`κ(XᵀX) = κ(X)²`

A backward-stable solve returns an answer whose relative error scales with the condition number times machine epsilon (`ε ≈ 2.2e-16`). QR works on `X` directly, so its error scales with `κ(X)·ε`. The normal equations work on `XᵀX`, so theirs scales with `κ(X)²·ε`. Same problem, same data, error squared.

**3. Do not compute what you are going to throw away.** A full SVD of an `m × n` matrix costs `O(mn·min(m, n))`. If you only want the top `k` singular triplets, randomized SVD costs `O(mnk)` and Lanczos costs `O(k·nnz(A))`. For MovieLens-shaped data — 943 × 1682 and 5.7% observed — `nnz` is about 90,000 against `mn ≈ 1.6` million, and the gap between those two costs is the entire reason large-scale recommenders are feasible.

> 🇪🇸 Estos son conteos de flops, no medidas: el término dominante de la aritmética que cada algoritmo realiza sobre una matriz densa `m × n` con `m ≥ n`, tomados de Trefethen & Bau. Ya están cargados como `FLOPS` en la celda de preparación.
>
> Tres consecuencias:
>
> **1. Factoriza una vez, resuelve muchas.** Cholesky cuesta `n³/3` una vez; cada solución posterior son dos sustituciones triangulares a `O(n²)`. Así que `m` lados derechos cuestan `O(n³ + mn²)`, **no** `O(mn³)`.
>
> **2. Las ecuaciones normales elevan al cuadrado el número de condición**, porque `κ(XᵀX) = κ(X)²`. El error de QR escala con `κ(X)·ε`; el de las ecuaciones normales, con `κ(X)²·ε`.
>
> **3. No calcules lo que vas a tirar.** Una SVD completa cuesta `O(mn·min(m, n))`; si solo quieres las primeras `k` componentes, la SVD aleatorizada cuesta `O(mnk)` y Lanczos `O(k·nnz(A))`.

## Exercise 1 — least squares two ways / Ejercicio 1 — mínimos cuadrados de dos maneras

We fit a **degree-10 polynomial** to the real airline series: 144 monthly passenger counts, with time rescaled to `[0, 1]` so the design matrix is as well behaved as a Vandermonde ever gets.

`X = np.vander(month_scaled, 11, increasing=True)`  →  shape `(144, 11)`

That is a real, ordinary modelling choice — nobody constructed it to be pathological — and it is already badly conditioned. Fit it two ways:

- **Normal equations:** `β = solve(XᵀX, Xᵀy)`
- **QR:** `Q, R = qr(X)`, then `β = solve(R, Qᵀy)`

Use `np.linalg.lstsq` (which is SVD-based, and the most stable of the three) as the reference answer.

### The cost side, before the stability side / El coste, antes de la estabilidad

It matters that the normal equations are not merely the naive choice — they are the **cheaper** one. Forming `XᵀX` and solving it costs about `mn² + n³/3` flops; Householder QR costs `2mn² − 2n³/3`. For a tall matrix, where `m ≫ n`, that is roughly **half the work**.

So this is a real trade, not a free lunch: you are buying `κ(X)` instead of `κ(X)²` with a factor of two in flops. On a 144 × 11 problem those flops are microseconds and the choice is obvious. The exercise is worth doing because on a matrix where the flops *do* matter, the temptation is real — and the error it buys you does not announce itself.

### Predict before you run / Predice antes de ejecutar

Both methods minimize the same quantity. The **residual** `‖Xβ − y‖` will look nearly identical. The **coefficients** will not. Which comparison tells you the truth about stability?

> 🇪🇸 Ajustamos un **polinomio de grado 10** a la serie real de aerolíneas: 144 meses, con el tiempo reescalado a `[0, 1]` para que la matriz de diseño esté lo mejor condicionada que una Vandermonde puede estar. Es una elección de modelado normal, y ya está mal condicionada.
>
> Ajústala de dos maneras — ecuaciones normales y QR — y usa `np.linalg.lstsq` (basada en SVD) como respuesta de referencia.
>
> **El coste, antes de la estabilidad.** Las ecuaciones normales no son solo la opción ingenua: son la **más barata**. Formar `XᵀX` y resolverlo cuesta unos `mn² + n³/3` flops; la QR de Householder cuesta `2mn² − 2n³/3`. Con `m ≫ n`, eso es aproximadamente **la mitad del trabajo**. Es un intercambio real: compras `κ(X)` en vez de `κ(X)²` a cambio de un factor de dos en flops.
>
> **Predice antes de ejecutar:** ambos métodos minimizan lo mismo, así que el **residuo** será casi idéntico. Los **coeficientes** no. ¿Cuál de las dos comparaciones dice la verdad sobre la estabilidad?

In [ ]:
# TODO 1 / TAREA 1
#
# EN:
# 1. Build the design matrix:
#       X = np.vander(month_scaled, 11, increasing=True)
#    and set y = passengers.
# 2. Print np.linalg.cond(X) and np.linalg.cond(X.T @ X).
#    Confirm the second is roughly the square of the first.
# 3. Normal equations:  beta_normal = np.linalg.solve(X.T @ X, X.T @ y)
# 4. QR:                Q, R = np.linalg.qr(X)
#                       beta_qr = np.linalg.solve(R, Q.T @ y)
# 5. Reference:         beta_ref = np.linalg.lstsq(X, y, rcond=None)[0]
# 6. Print the residual ||X @ beta - y|| for all three. Nearly identical?
# 7. Print the RELATIVE COEFFICIENT ERROR of each against beta_ref.
#    That is where the two methods part company.
#
# ES:
# 1. Construye la matriz de diseño de grado 10 y define y = passengers.
# 2. Imprime cond(X) y cond(X.T @ X); comprueba que la segunda es
#    aproximadamente el cuadrado de la primera.
# 3. Resuelve con las ecuaciones normales.
# 4. Resuelve con QR.
# 5. Toma np.linalg.lstsq como referencia.
# 6. Imprime el residuo de las tres. ¿Casi idénticos?
# 7. Imprime el error relativo de los coeficientes frente a la referencia.

In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo primero { display-mode: 'form' }

X = np.vander(month_scaled, 11, increasing=True)
y = passengers

cond_X = np.linalg.cond(X)
cond_normal = np.linalg.cond(X.T @ X)

beta_normal = np.linalg.solve(X.T @ X, X.T @ y)

Q, R = np.linalg.qr(X)
beta_qr = np.linalg.solve(R, Q.T @ y)

beta_ref = np.linalg.lstsq(X, y, rcond=None)[0]

eps = np.finfo(float).eps

print("Design matrix / Matriz de diseño:", X.shape)
print("cond(X)      =", f"{cond_X:.3e}")
print("cond(XtX)    =", f"{cond_normal:.3e}",
      "   cond(X)^2 =", f"{cond_X ** 2:.3e}")
print()
print("Predicted error floor / Cota de error predicha:")
print("   QR             kappa(X)   * eps =", f"{cond_X * eps:.3e}")
print("   normal eqs.    kappa(X)^2 * eps =", f"{cond_X ** 2 * eps:.3e}")
print()

residual = lambda b: np.linalg.norm(X @ b - y)

print("Residual ||X beta - y|| / Residuo:")
print("   normal equations / ecuaciones normales:", f"{residual(beta_normal):.6f}")
print("   QR                                    :", f"{residual(beta_qr):.6f}")
print("   lstsq (reference / referencia)        :", f"{residual(beta_ref):.6f}")
print()

coef_error = lambda b: (np.linalg.norm(b - beta_ref)
                        / np.linalg.norm(beta_ref))

print("Relative coefficient error / Error relativo de coeficientes:")
print("   normal equations / ecuaciones normales:", f"{coef_error(beta_normal):.3e}")
print("   QR                                    :", f"{coef_error(beta_qr):.3e}")
print()
print("EN: the residuals agree to six decimals, so residual comparison hides")
print("    the problem entirely. The coefficients differ by about eleven")
print("    orders of magnitude, and that gap is kappa(X)^2 against kappa(X).")
print("ES: los residuos coinciden en seis decimales, así que comparar residuos")
print("    oculta el problema por completo. Los coeficientes difieren en unos")
print("    once órdenes de magnitud: esa brecha es kappa(X)^2 frente a kappa(X).")

### Interactive conditioning explorer / Explorador interactivo de condicionamiento

Slide the polynomial degree and watch four numbers move together on the real airline data:

- `κ(X)` — the condition number of the design matrix;
- `κ(XᵀX)` — which tracks `κ(X)²`, not `κ(X)`;
- the coefficient error of the **normal equations**;
- the coefficient error of **QR**.

Degree 4 is harmless. By degree 10 the normal equations have lost eleven digits that QR still has. By degree 12 they return a visibly worse *fit*, not just worse coefficients — the residual itself degrades, which is the point at which even a residual-only check would finally notice.

> 🇪🇸 Desliza el grado del polinomio y observa cómo se mueven juntos cuatro números sobre los datos reales de aerolíneas: `κ(X)`, `κ(XᵀX)` (que sigue a `κ(X)²`), y el error de coeficientes de las ecuaciones normales y de QR.
>
> El grado 4 es inofensivo. En el grado 10 las ecuaciones normales han perdido once dígitos que QR conserva. En el grado 12 empeora el propio residuo, que es cuando incluso una comprobación basada solo en residuos se daría cuenta.

In [ ]:
# The degree-10 design matrix and target (Exercise 1, TODO 1 step 1), rebound
# here so this explorer runs whether or not the folded solution was executed.
X = np.vander(month_scaled, 11, increasing=True)
y = passengers

degree_slider = widgets.IntSlider(
    value=10,
    min=2,
    max=14,
    step=1,
    description="Degree / Grado:",
    continuous_update=False,
    style={"description_width": "140px"},
)

def compare_conditioning(degree):
    Xd = np.vander(month_scaled, degree + 1, increasing=True)

    cond_X = np.linalg.cond(Xd)
    cond_normal = np.linalg.cond(Xd.T @ Xd)

    beta_normal = np.linalg.solve(Xd.T @ Xd, Xd.T @ y)
    Qd, Rd = np.linalg.qr(Xd)
    beta_qr = np.linalg.solve(Rd, Qd.T @ y)
    beta_ref = np.linalg.lstsq(Xd, y, rcond=None)[0]

    err = lambda b: (np.linalg.norm(b - beta_ref)
                     / np.linalg.norm(beta_ref))
    residual = lambda b: np.linalg.norm(Xd @ b - y)

    print("Design matrix / Matriz de diseño:", Xd.shape)
    print("cond(X)   =", f"{cond_X:.2e}")
    print("cond(XtX) =", f"{cond_normal:.2e}",
          "   cond(X)^2 =", f"{cond_X ** 2:.2e}")
    print()
    print("Coefficient error / Error de coeficientes:")
    print("   normal equations / ecuaciones normales:", f"{err(beta_normal):.2e}")
    print("   QR                                    :", f"{err(beta_qr):.2e}")
    print()
    print("Residual / Residuo:")
    print("   normal equations / ecuaciones normales:", f"{residual(beta_normal):.4f}")
    print("   QR                                    :", f"{residual(beta_qr):.4f}")

    fig, ax = plt.subplots(figsize=(7.5, 3.2))
    ax.plot(month, y, ".", color="0.55", markersize=4,
            label="Real passengers / Pasajeros reales")
    ax.plot(month, Xd @ beta_qr, "-", linewidth=2, label="QR")
    ax.plot(month, Xd @ beta_normal, "--", linewidth=1.6,
            label="Normal equations / Ecuaciones normales")
    ax.set_xlabel("Month index / Índice de mes")
    ax.set_ylabel("Passengers / Pasajeros")
    ax.set_title(f"Degree {degree} polynomial fit / Ajuste polinómico de grado {degree}")
    ax.legend(fontsize=8)
    plt.show()

conditioning_output = widgets.interactive_output(
    compare_conditioning,
    {"degree": degree_slider},
)

display(widgets.VBox([degree_slider, conditioning_output]))

## Exercise 2 — factor once, solve many / Ejercicio 2 — factoriza una vez, resuelve muchas

This is the `O(n³ + mn²)` claim from §12.2, on a real system.

We build the **RBF kernel matrix** of the 1797 digit images: `G[i, j] = exp(−γ‖xᵢ − xⱼ‖²)`, plus a small ridge term on the diagonal. That is a genuine 1797 × 1797 symmetric positive-definite system — it is exactly the linear algebra behind kernel ridge regression — and solving it against one-hot digit labels gives a classifier that is about 98% accurate on held-out digits.

Then solve it three ways and time all three:

1. **Factor once**, solve every right-hand side against the stored factor.
2. **Explicit inverse**: `np.linalg.inv(G) @ B`.
3. **One `np.linalg.solve` call per column** — refactoring from scratch every time.

### Predict before you run / Predice antes de ejecutar

Method 3 does `m` full factorizations where method 1 does one. With `m = 200`, how many times slower should it be? And is the explicit inverse merely slower, or *also* less accurate?

> 🇪🇸 Esta es la afirmación `O(n³ + mn²)` de §12.2, sobre un sistema real.
>
> Construimos la **matriz kernel RBF** de las 1797 imágenes de dígitos más un pequeño término de regularización en la diagonal: un sistema simétrico definido positivo real de 1797 × 1797, el álgebra lineal exacta detrás de la regresión ridge con kernel, y que clasifica dígitos no vistos con un 98% de acierto.
>
> Resuélvelo de tres formas y cronométralas: factorizando una vez, con la inversa explícita, y con una llamada a `solve` por columna.
>
> **Predice antes de ejecutar:** el método 3 hace `m` factorizaciones completas donde el 1 hace una. Con `m = 200`, ¿cuántas veces más lento debería ser? ¿Y la inversa explícita es solo más lenta, o *también* menos precisa?

In [ ]:
# TODO 2 / TAREA 2
#
# EN:
# 1. Build the RBF kernel matrix of the digits and add a ridge term:
#       sq = np.sum(D ** 2, axis=1)
#       D2 = sq[:, None] + sq[None, :] - 2 * D @ D.T
#       G  = np.exp(-0.05 * np.maximum(D2, 0)) + 1e-6 * np.eye(len(D))
#    Print G.shape and np.linalg.cond(G).
# 2. Build 200 right-hand sides: B = rng.standard_normal((len(G), 200)).
# 3. Time, with best_time(...):
#       a) cho_solve(cho_factor(G), B)                 <- factor once
#       b) np.linalg.inv(G) @ B                        <- explicit inverse
#       c) one np.linalg.solve(G, B[:, j]) per column  <- refactor each time
#    Use repeats=1 for (c); it is slow on purpose.
# 4. Print each time and the ratio against (a).
# 5. Print the relative residual ||G @ Xhat - B|| / ||B|| for (a) and (b).
#    Which is more accurate?
# 6. Bonus: solve against one-hot digit labels instead of random noise and
#    report the training accuracy of the resulting kernel-ridge classifier.
#
# ES:
# 1. Construye la matriz kernel RBF de los dígitos más un término ridge.
# 2. Crea 200 lados derechos aleatorios.
# 3. Cronometra las tres estrategias (usa repeats=1 en la tercera).
# 4. Imprime cada tiempo y la razón frente a la primera.
# 5. Imprime el residuo relativo de las dos primeras. ¿Cuál es más precisa?
# 6. Extra: resuelve contra etiquetas one-hot y reporta la exactitud.

In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo primero { display-mode: 'form' }

sq = np.sum(D ** 2, axis=1)
D2 = sq[:, None] + sq[None, :] - 2 * D @ D.T
G = np.exp(-0.05 * np.maximum(D2, 0)) + 1e-6 * np.eye(len(D))

n_rhs = 200
B = rng.standard_normal((len(G), n_rhs))

print("Kernel system / Sistema kernel:", G.shape)
print("cond(G) =", f"{np.linalg.cond(G):.3e}")
print("Right-hand sides / Lados derechos:", n_rhs)
print()

t_factor = best_time(lambda: cho_solve(cho_factor(G), B))
t_inverse = best_time(lambda: np.linalg.inv(G) @ B)
t_percol = best_time(
    lambda: np.column_stack(
        [np.linalg.solve(G, B[:, j]) for j in range(n_rhs)]),
    repeats=1,
)

print("Factor once + 200 solves / Factorizar una vez + 200 soluciones:",
      f"{t_factor * 1e3:9.1f} ms   (1.0x)")
print("Explicit inverse / Inversa explícita:                          ",
      f"{t_inverse * 1e3:9.1f} ms   ({t_inverse / t_factor:.1f}x)")
print("One solve per column / Una solución por columna:               ",
      f"{t_percol * 1e3:9.1f} ms   ({t_percol / t_factor:.1f}x)")
print()

X_factor = cho_solve(cho_factor(G), B)
X_inverse = np.linalg.inv(G) @ B
rel = lambda Xh: np.linalg.norm(G @ Xh - B) / np.linalg.norm(B)

print("Relative residual / Residuo relativo:")
print("   factor once / factorizar una vez:", f"{rel(X_factor):.3e}")
print("   explicit inverse / inversa explícita:", f"{rel(X_inverse):.3e}")
print()

# Score it held out: with this little ridge the solve interpolates, so a
# training accuracy of 1.0 would say nothing about the model.
holdout = np.random.default_rng(7).permutation(len(D))
fit_idx, score_idx = holdout[:1200], holdout[1200:]
alpha = cho_solve(cho_factor(G[np.ix_(fit_idx, fit_idx)]),
                  np.eye(10)[digit_labels[fit_idx]])
predicted = (G[np.ix_(score_idx, fit_idx)] @ alpha).argmax(axis=1)
accuracy = np.mean(predicted == digit_labels[score_idx])
print("Kernel-ridge held-out accuracy / Exactitud reservada:",
      f"{accuracy:.4f}", f"on / sobre {len(score_idx)} unseen digits")
print()
print("EN: one factorization amortized over 200 solves is O(n^3 + m n^2).")
print("    Refactoring per column is O(m n^3) and the measured ratio shows it.")
print("    The explicit inverse is both slower and less accurate — it is never")
print("    the right call.")
print("ES: una factorización amortizada en 200 soluciones es O(n^3 + m n^2);")
print("    refactorizar por columna es O(m n^3), y la razón medida lo muestra.")
print("    La inversa explícita es más lenta Y menos precisa: nunca es la")
print("    llamada correcta.")

### Interactive repeated-solve explorer / Explorador interactivo de resolución repetida

Slide the number of right-hand sides and read the three strategies against each other on the same real 1797 × 1797 kernel system.

Two things to watch:

- **Where the lines cross.** At `m = 1` there is nothing to amortize and the three are within a small factor of each other. The gap opens with `m`, because only one of the three curves has an `m`-independent term.
- **How badly the flop count under-sells the block solve.** Solving 200 right-hand sides at once is far faster than 200 separate triangular solves, because LAPACK turns the block version into matrix–matrix products. Flop counts predict the *shape* of these curves; they do not predict the constants, which is precisely why §12.3 measures.

The per-column curve is measured up to `m = 20` and extended as a straight line beyond it — that extension is safe because refactoring per column really is exactly linear in `m` (measured within 2% at `m = 200`).

> 🇪🇸 Desliza el número de lados derechos y compara las tres estrategias sobre el mismo sistema kernel real de 1797 × 1797.
>
> Fíjate en **dónde se cruzan las líneas** — con `m = 1` no hay nada que amortizar — y en **cuánto subestima el conteo de flops la resolución por bloques**: resolver 200 lados derechos a la vez es mucho más rápido que 200 sustituciones triangulares separadas, porque LAPACK lo convierte en productos matriz–matriz. Los flops predicen la *forma* de estas curvas, no las constantes.
>
> La curva por columna se mide hasta `m = 20` y se extiende como recta: refactorizar por columna es exactamente lineal en `m` (verificado con un 2% de error en `m = 200`).

In [ ]:
# The real kernel system (Exercise 2, TODO 2 step 1), rebound here so this
# explorer runs whether or not the folded solution was executed.
sq = np.sum(D ** 2, axis=1)
D2 = sq[:, None] + sq[None, :] - 2 * D @ D.T
G = np.exp(-0.05 * np.maximum(D2, 0)) + 1e-6 * np.eye(len(D))

rhs_grid = np.array([1, 2, 5, 10, 20, 50, 100, 200])
MEASURED_PER_COLUMN_UP_TO = 20

t_factor_once, t_inverse_grid, t_per_column = [], [], []

for m_rhs in rhs_grid:
    B = rng.standard_normal((len(G), int(m_rhs)))
    t_factor_once.append(best_time(lambda: cho_solve(cho_factor(G), B)))
    t_inverse_grid.append(best_time(lambda: np.linalg.inv(G) @ B))
    if m_rhs <= MEASURED_PER_COLUMN_UP_TO:
        t_per_column.append(best_time(
            lambda: np.column_stack(
                [np.linalg.solve(G, B[:, j]) for j in range(int(m_rhs))]),
            repeats=1,
        ))

t_factor_once = np.array(t_factor_once)
t_inverse_grid = np.array(t_inverse_grid)

# Refactoring per column is exactly linear in m, so one slope extends it.
per_column_slope = (np.array(t_per_column)
                    / rhs_grid[:len(t_per_column)]).mean()
t_per_column = per_column_slope * rhs_grid

print("Measured on / Medido sobre:", G.shape)
print("Per-column cost per solve / Coste por solución:",
      f"{per_column_slope * 1e3:.1f} ms")
print()

rhs_slider = widgets.SelectionSlider(
    options=[int(m_rhs) for m_rhs in rhs_grid],
    value=200,
    description="Right-hand sides / Lados derechos:",
    continuous_update=False,
    style={"description_width": "220px"},
)

def show_repeated_solve(m_rhs):
    i = int(np.where(rhs_grid == m_rhs)[0][0])
    base = t_factor_once[i]

    print(f"m = {m_rhs} right-hand sides / lados derechos")
    print("Factor once / Factorizar una vez:  ",
          f"{base * 1e3:9.1f} ms   (1.0x)")
    print("Explicit inverse / Inversa explícita:",
          f"{t_inverse_grid[i] * 1e3:9.1f} ms   "
          f"({t_inverse_grid[i] / base:.1f}x)")
    print("Refactor per column / Refactorizar por columna:",
          f"{t_per_column[i] * 1e3:9.1f} ms   "
          f"({t_per_column[i] / base:.1f}x)"
          + ("" if m_rhs <= MEASURED_PER_COLUMN_UP_TO
             else "   [extrapolated / extrapolado]"))

    fig, ax = plt.subplots(figsize=(7.0, 3.4))
    ax.loglog(rhs_grid, t_factor_once * 1e3, "o-",
              label="Factor once / Factorizar una vez")
    ax.loglog(rhs_grid, t_inverse_grid * 1e3, "s-",
              label="Explicit inverse / Inversa explícita")
    ax.loglog(rhs_grid, t_per_column * 1e3, "^--",
              label="Refactor per column / Por columna")
    ax.axvline(m_rhs, color="0.6", linewidth=1)
    ax.set_xlabel("Right-hand sides m / Lados derechos m")
    ax.set_ylabel("Time (ms) / Tiempo (ms)")
    ax.set_title("Cost of m solves on one real 1797x1797 SPD system")
    ax.legend(fontsize=8)
    ax.grid(True, which="both", alpha=0.25)
    plt.show()

repeated_output = widgets.interactive_output(
    show_repeated_solve,
    {"m_rhs": rhs_slider},
)

display(widgets.VBox([rhs_slider, repeated_output]))

## 12.3 The cost table, measured / La tabla de costes, medida

§12.2 asserted a table of exponents. This section makes you produce it.

The idea is simple. If `t(n) = C·nᵖ`, then `log t = log C + p·log n` — a straight line whose **slope is the exponent**. So: time each factorization across a sweep of sizes, fit a line through the log-log points, and compare the slope you measured against the exponent you were told.

### What you should expect to see / Qué deberías ver

Not 3.0. Fitted slopes on a modern laptop or a Colab CPU usually land between about **2.2 and 2.9**, and that is not an error in the theory. Two things pull the measurement below the count:

- **Parallelism grows with `n`.** A 128 × 128 factorization cannot keep multiple cores busy; a 1024 × 1024 one can. The machine gets more efficient as the problem grows, which flattens the curve.
- **Cache reuse grows with `n`.** Blocked LAPACK algorithms turn more of the work into matrix–matrix products as the blocks get bigger.

Both effects shrink as `n` keeps growing, so the fitted slope creeps *up* toward 3 with larger sizes. What is robust — and what you should actually check — is the **ordering and the ratios between methods at a fixed `n`**. The flop table predicts a full SVD costs about `13n³ / (n³/3) = 39×` a Cholesky. Measure that ratio; it holds far better than either exponent alone.

> 🇪🇸 §12.2 afirmó una tabla de exponentes; esta sección te hace producirla.
>
> Si `t(n) = C·nᵖ`, entonces `log t = log C + p·log n`: una recta cuya **pendiente es el exponente**. Cronometra cada factorización sobre un barrido de tamaños, ajusta una recta a los puntos log-log y compara la pendiente medida con el exponente que te dijeron.
>
> **No esperes 3.0.** Las pendientes ajustadas suelen quedar entre **2.2 y 2.9**, y no es un error de la teoría: el paralelismo y la reutilización de caché crecen con `n`, así que la máquina se vuelve más eficiente conforme crece el problema y la curva se aplana. Ambos efectos se reducen al seguir creciendo `n`, de modo que la pendiente sube hacia 3.
>
> Lo robusto es **el orden y las razones entre métodos a `n` fijo**: la tabla predice que una SVD completa cuesta unas `39×` una Cholesky. Esa razón se sostiene mucho mejor que cualquier exponente por separado.

In [ ]:
# The size sweep. On a Colab CPU this takes roughly half a minute; drop the
# largest size if you are impatient.
sweep_sizes = np.array([128, 192, 256, 384, 512, 768])

# Its own generator, so re-running any cell above cannot change the matrices
# this sweep is timed on — the quoted ratios depend on them.
sweep_rng = np.random.default_rng(1)

def make_spd(n):
    """A symmetric positive-definite n x n matrix, so every method applies."""
    A = sweep_rng.standard_normal((n, n))
    return A @ A.T + n * np.eye(n)

sweep_matrices = {int(n): make_spd(int(n)) for n in sweep_sizes}

SWEEP_METHODS = {
    "Cholesky": np.linalg.cholesky,
    "LU": lu_factor,
    "QR": np.linalg.qr,
    "Eigendecomposition": np.linalg.eigh,
    "Thin SVD": lambda A: np.linalg.svd(A, full_matrices=False),
}

sweep_times = {}
for name, call in SWEEP_METHODS.items():
    sweep_times[name] = np.array([
        best_time(lambda: call(sweep_matrices[int(n)])) for n in sweep_sizes
    ])

print("Sizes / Tamaños:", [int(n) for n in sweep_sizes])
print()
header = "method / método (ms)".ljust(22) + "".join(
    f"{int(n):>10d}" for n in sweep_sizes)
print(header)
print("-" * len(header))
for name, times in sweep_times.items():
    print(name.ljust(22)
          + "".join(f"{t * 1e3:10.2f}" for t in times))
print()
largest = int(sweep_sizes[-1])
ratio = (sweep_times["Thin SVD"][-1] / sweep_times["Cholesky"][-1])
print(f"At n = {largest}: SVD / Cholesky = {ratio:.1f}x  "
      f"(flop table predicts / la tabla predice 39x)")

## Exercise 3 — fit the exponent / Ejercicio 3 — ajusta el exponente

You now hold `sweep_sizes` and `sweep_times`. Turn them into exponents.

### Predict before you run / Predice antes de ejecutar

Which method's fitted slope will be *closest* to 3, and why? Think about which one does the most arithmetic per byte moved — that is the one whose timing is least polluted by memory traffic.

> 🇪🇸 Ya tienes `sweep_sizes` y `sweep_times`. Conviértelos en exponentes.
>
> **Predice antes de ejecutar:** ¿qué método tendrá la pendiente más cercana a 3 y por qué? Piensa cuál hace más aritmética por byte movido: ese es el que menos contamina el tráfico de memoria.

In [ ]:
# TODO 3 / TAREA 3
#
# EN:
# 1. For each method in sweep_times, fit a line through the log-log points:
#       slope, intercept = np.polyfit(np.log(sweep_sizes), np.log(times), 1)
#    The slope is the measured exponent.
# 2. Print the fitted exponent beside the predicted one. Every dense
#    factorization here is cubic, so the predicted exponent is 3 for all five.
# 3. Draw a log-log plot: the measured points, the fitted line, and a
#    reference line proportional to n**3 through the first point.
# 4. Compute the measured cost ratio SVD / Cholesky at the largest size and
#    compare it against the flop-table prediction of 39x.
# 5. Answer in one sentence: which is the more trustworthy prediction from
#    the flop table on this machine, the exponent or the ratio?
#
# ES:
# 1. Ajusta una recta a los puntos log-log de cada método; la pendiente es
#    el exponente medido.
# 2. Imprime el exponente ajustado junto al predicho (3 para los cinco).
# 3. Dibuja un gráfico log-log con los puntos, la recta ajustada y una
#    referencia proporcional a n**3.
# 4. Calcula la razón medida SVD / Cholesky en el mayor tamaño y compárala
#    con las 39x que predice la tabla de flops.
# 5. Responde en una frase: en esta máquina, ¿qué predice mejor la tabla de
#    flops, el exponente o la razón?

In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo primero { display-mode: 'form' }

log_n = np.log(sweep_sizes)

print("method / método".ljust(20), "fitted / ajustado", " predicted / predicho")
print("-" * 60)
fitted = {}
for name, times in sweep_times.items():
    slope, intercept = np.polyfit(log_n, np.log(times), 1)
    fitted[name] = (slope, intercept)
    print(name.ljust(20), f"{slope:16.2f}", f"{3.0:20.1f}")

fig, ax = plt.subplots(figsize=(7.5, 4.2))
for name, times in sweep_times.items():
    slope, intercept = fitted[name]
    line = ax.loglog(sweep_sizes, times * 1e3, "o", label=f"{name} (p={slope:.2f})")
    ax.loglog(sweep_sizes, np.exp(intercept) * sweep_sizes ** slope * 1e3,
              "-", linewidth=1, color=line[0].get_color())

reference = sweep_times["Thin SVD"][0] * (sweep_sizes / sweep_sizes[0]) ** 3
ax.loglog(sweep_sizes, reference * 1e3, "k:", linewidth=1.5,
          label="slope 3 reference / referencia pendiente 3")
ax.set_xlabel("Matrix size n / Tamaño n")
ax.set_ylabel("Time (ms) / Tiempo (ms)")
ax.set_title("Measured cost against size / Coste medido frente al tamaño")
ax.legend(fontsize=8)
ax.grid(True, which="both", alpha=0.25)
plt.show()

measured_ratio = sweep_times["Thin SVD"][-1] / sweep_times["Cholesky"][-1]
print()
print(f"SVD / Cholesky at n = {int(sweep_sizes[-1])}: "
      f"measured / medido {measured_ratio:.1f}x, predicted / predicho 39x")
print()
print("EN: the fitted exponents come in under 3 because parallelism and cache")
print("    reuse both improve as n grows — the machine gets faster at the same")
print("    work. The RATIO between methods is the more trustworthy prediction:")
print("    it cancels the machine out, because both methods gain from the same")
print("    hardware effects.")
print("ES: los exponentes ajustados quedan por debajo de 3 porque el")
print("    paralelismo y la caché mejoran al crecer n: la máquina se vuelve más")
print("    eficiente con el mismo trabajo. La RAZÓN entre métodos predice")
print("    mejor, porque cancela la máquina: ambos métodos ganan lo mismo.")

## 12.4 SVD — the best rank-k there is / SVD — la mejor aproximación de rango k

Of the six factorizations, exactly one comes with an optimality guarantee, and it is worth stating precisely.

**Eckart–Young–Mirsky.** Let `A = UΣVᵀ` and let `A_k = Σ_{i≤k} σᵢ uᵢ vᵢᵀ` be the truncation to the top `k` singular triplets. Then for *every* matrix `B` with `rank(B) ≤ k`:

`‖A − A_k‖_F ≤ ‖A − B‖_F`,  and  `‖A − A_k‖_F = √(Σ_{i>k} σᵢ²)`

Two claims, and both are checkable. The first says no rank-`k` matrix produced by any method — NMF, a neural autoencoder, a clever hand-picked basis — gets closer in Frobenius norm. The second says you do not need to build `A_k` to know its error: the singular values you already have tell you exactly.

The next cell checks the equality on the real 512 × 512 astronaut image, to machine precision.

> 🇪🇸 De las seis factorizaciones, exactamente una viene con garantía de optimalidad.
>
> **Eckart–Young–Mirsky.** Si `A = UΣVᵀ` y `A_k` es la truncación a las primeras `k` componentes, entonces para *toda* matriz `B` con `rank(B) ≤ k` se cumple `‖A − A_k‖_F ≤ ‖A − B‖_F`, y además `‖A − A_k‖_F = √(Σ_{i>k} σᵢ²)`.
>
> Dos afirmaciones, ambas comprobables: ninguna matriz de rango `k` producida por ningún método se acerca más, y no necesitas construir `A_k` para conocer su error — los valores singulares ya te lo dicen exactamente.
>
> La celda siguiente comprueba la igualdad sobre la imagen real de 512 × 512, con precisión de máquina.

In [ ]:
image = IMAGES["astronaut"]

U_img, S_img, Vt_img = np.linalg.svd(image, full_matrices=False)

def truncate(k, U=U_img, S=S_img, Vt=Vt_img):
    """The rank-k truncation A_k = sum_{i<k} sigma_i u_i v_i^T."""
    return (U[:, :k] * S[:k]) @ Vt[:k]

print("Image / Imagen:", image.shape,
      " singular values / valores singulares:", S_img.shape[0])
print()
print(" k   ||A - A_k||_F    sqrt(tail)       difference / diferencia")
print("-" * 66)
for k in (5, 16, 20, 50, 100):
    built = np.linalg.norm(image - truncate(k))
    tail = np.sqrt(np.sum(S_img[k:] ** 2))
    print(f"{k:3d}   {built:13.9f}   {tail:13.9f}   {abs(built - tail):.3e}")

print()
print("EN: the two columns agree to within 1e-13. You never have to build")
print("    A_k to know how good it would be — the singular values said so.")
print("ES: las dos columnas coinciden dentro de 1e-13. Nunca hace falta")
print("    construir A_k para saberlo: los valores singulares ya lo")
print("    dijeron.")

### Interactive rank explorer on a real image / Explorador interactivo de rango sobre una imagen real

Slide the rank and watch three numbers that do *not* move together:

- **relative error** and **PSNR in dB** — how good it looks;
- **numbers stored**, `k(m + n + 1)` against `mn` — how much you kept;
- **bytes stored**, which depends on the dtype you save the factors in. This is the number people quote and get wrong. At `k = 16` the factors are 6.3% of the pixel *count*, but 12.5% of the raw *bytes* if you store them as `int16` and 25% as `float32`. A 512 × 512 `uint8` image is one byte per pixel; a singular vector entry is not.

Reference points on the astronaut image, which you can read off the widget: **rank 16 → about 21.1 dB at 12.5% of the raw bytes in `int16`**, and **rank 50 → about 27.4 dB at 39.1%**. By rank 128 the `int16` factors cost as much as the whole image did.

That last fact is the honest headline. Truncated SVD is a superb *analysis* tool and a mediocre image *codec* — JPEG at the same file size beats it comfortably — because JPEG exploits structure an SVD cannot see. Use rank truncation to find out what a matrix is made of, not to make files smaller.

> 🇪🇸 Desliza el rango y observa tres números que *no* se mueven juntos: el **error relativo** y el **PSNR en dB**; los **números almacenados**, `k(m + n + 1)` frente a `mn`; y los **bytes almacenados**, que dependen del tipo con que guardes los factores.
>
> Ese último es el que la gente cita mal: con `k = 16`, los factores son el 6.3% del *número* de píxeles, pero el 12.5% de los *bytes* en `int16` y el 25% en `float32`. Una imagen `uint8` es un byte por píxel; una entrada de un vector singular no.
>
> Puntos de referencia sobre la imagen astronaut: **rango 16 → unos 21.1 dB con el 12.5% de los bytes**, y **rango 50 → unos 27.4 dB con el 39.1%**. Hacia el rango 128, los factores en `int16` ocupan tanto como la imagen entera.
>
> Ese es el titular honesto: la SVD truncada es una herramienta de *análisis* excelente y un *códec* mediocre — JPEG gana con holgura al mismo tamaño de archivo. Usa la truncación para saber de qué está hecha una matriz, no para hacer archivos más pequeños.

In [ ]:
image_choice = widgets.ToggleButtons(
    options=[("Astronaut", "astronaut"), ("Camera", "camera")],
    value="astronaut",
    description="Image / Imagen:",
    style={"description_width": "130px"},
)

rank_slider = widgets.IntSlider(
    value=16,
    min=1,
    max=200,
    step=1,
    description="Rank k / Rango k:",
    continuous_update=False,
    style={"description_width": "130px"},
)

# One SVD per image, computed once so the slider stays instant.
IMAGE_SVD = {name: np.linalg.svd(A, full_matrices=False)
             for name, A in IMAGES.items()}

def explore_rank(which, k):
    A = IMAGES[which]
    U, S, Vt = IMAGE_SVD[which]
    m, n = A.shape

    Ak = (U[:, :k] * S[:k]) @ Vt[:k]

    stored = k * (m + n + 1)
    energy = np.sum(S[:k] ** 2) / np.sum(S ** 2)

    print("Rank / Rango:", k, " of / de", min(m, n))
    print("Relative error / Error relativo:", f"{relative_error(A, Ak):.4f}")
    print("PSNR:", f"{psnr(A, Ak):.2f} dB")
    print("Energy kept / Energía conservada:", f"{100 * energy:.3f}%")
    print()
    print("Numbers stored / Números almacenados:", f"{stored:,}",
          "of / de", f"{m * n:,}", f"({100 * stored / (m * n):.2f}%)")
    print("Bytes as int16 / Bytes en int16:",
          f"{100 * stored * 2 / (m * n):.1f}% of the raw uint8 image")
    print("Bytes as float32 / Bytes en float32:",
          f"{100 * stored * 4 / (m * n):.1f}% of the raw uint8 image")

    fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.6))
    axes[0].imshow(A, cmap="gray", vmin=0, vmax=1)
    axes[0].set_title("Original")
    axes[1].imshow(Ak, cmap="gray", vmin=0, vmax=1)
    axes[1].set_title(f"Rank {k} / Rango {k}")
    axes[2].imshow(np.abs(A - Ak), cmap="magma")
    axes[2].set_title("|difference| / |diferencia|")
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

rank_output = widgets.interactive_output(
    explore_rank,
    {"which": image_choice, "k": rank_slider},
)

display(widgets.VBox([image_choice, rank_slider, rank_output]))

## Exercise 4 — the rank that crosses a threshold / Ejercicio 4 — el rango que cruza un umbral

"Rank 20" is not a decision anybody can defend. "The smallest rank that holds 25 dB" is.

Turn the requirement round: pick a quality target, then find the cheapest rank that meets it — and report what that rank actually costs to store.

### Predict before you run / Predice antes de ejecutar

Each extra 6 dB is roughly a halving of the error. Does the rank needed also double per 6 dB, or does it grow faster? The singular-value decay curve tells you before you measure.

> 🇪🇸 "Rango 20" no es una decisión defendible; "el menor rango que alcanza 25 dB" sí lo es.
>
> Da la vuelta al requisito: elige un objetivo de calidad, encuentra el rango más barato que lo cumple y reporta lo que ese rango cuesta almacenar.
>
> **Predice antes de ejecutar:** cada 6 dB extra es aproximadamente reducir el error a la mitad. ¿El rango necesario también se duplica cada 6 dB o crece más rápido? La curva de decaimiento de los valores singulares lo dice antes de medir.

In [ ]:
# TODO 4 / TAREA 4
#
# EN:
# 1. Write a function needed_rank(image, target_db) that returns the smallest
#    k with psnr(image, rank-k truncation) >= target_db.
#    Do it WITHOUT rebuilding the truncation for every k: Eckart-Young gives
#    the squared error directly as sum(S[k:] ** 2), so
#       mse(k) = sum(S[k:] ** 2) / (m * n)
#    and PSNR follows from that. One SVD, no loop over reconstructions.
# 2. Run it on the astronaut image for 20, 25, 30 and 35 dB.
# 3. For each, print the rank, the numbers stored k*(m + n + 1), and the
#    percentage of mn that represents.
# 4. Verify one of your answers by actually building the truncation and
#    calling psnr on it.
# 5. Plot required rank against target dB. Is the growth linear?
#
# ES:
# 1. Escribe needed_rank(image, target_db) que devuelva el menor k con
#    psnr >= target_db, SIN reconstruir para cada k: por Eckart-Young el
#    error cuadrático es sum(S[k:] ** 2), así que mse(k) = eso / (m * n).
# 2. Ejecútala sobre la imagen astronaut para 20, 25, 30 y 35 dB.
# 3. Para cada una, imprime el rango, los números almacenados y su
#    porcentaje frente a mn.
# 4. Verifica una respuesta construyendo la truncación y llamando a psnr.
# 5. Grafica el rango necesario frente al objetivo en dB. ¿Es lineal?

In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo primero { display-mode: 'form' }

def needed_rank(A, target_db, peak=1.0):
    """Smallest k reaching target_db, read straight off the singular values."""
    m, n = A.shape
    S = np.linalg.svd(A, compute_uv=False)
    # Eckart-Young: ||A - A_k||_F^2 = sum(S[k:] ** 2), so the MSE of the
    # rank-k truncation is that divided by the number of pixels.
    tail = np.concatenate([np.cumsum(S[::-1] ** 2)[::-1], [0.0]])
    mse = tail / (m * n)
    with np.errstate(divide="ignore"):
        db = 10 * np.log10(peak ** 2 / mse)
    return int(np.argmax(db >= target_db))

targets = [20, 25, 30, 35]
m_a, n_a = image.shape
ranks = [needed_rank(image, t) for t in targets]

print("target dB / objetivo   rank / rango   numbers stored / números   % of mn")
print("-" * 72)
for t, k in zip(targets, ranks):
    stored = k * (m_a + n_a + 1)
    print(f"{t:12d}   {k:14d}   {stored:22,}   {100 * stored / (m_a * n_a):6.2f}%")

check_k = ranks[1]
print()
print(f"Verification / Verificación at k = {check_k}:",
      f"{psnr(image, truncate(check_k)):.2f} dB",
      f"(target / objetivo {targets[1]} dB)")
print(f"One rank lower / Un rango menos, k = {check_k - 1}:",
      f"{psnr(image, truncate(check_k - 1)):.2f} dB")

fig, ax = plt.subplots(figsize=(6.5, 3.4))
ax.plot(targets, ranks, "o-")
ax.set_xlabel("Target PSNR (dB) / Objetivo")
ax.set_ylabel("Smallest rank / Rango mínimo")
ax.set_title("Quality is cheap at first and expensive later")
ax.grid(alpha=0.25)
plt.show()

print()
print("EN: the rank needed grows faster than linearly in dB. The singular")
print("    values decay quickly at first and then flatten, so the last few dB")
print("    cost more rank than the first twenty did.")
print("ES: el rango necesario crece más que linealmente en dB. Los valores")
print("    singulares decaen rápido y luego se aplanan, así que los últimos dB")
print("    cuestan más rango que los veinte primeros.")

## 12.5 Rank that moves a real metric / Rango que mueve una métrica real

Reconstruction error is a proxy. Nobody ships reconstruction error.

So ask the question that actually decides a rank: **at what `k` does the thing you built the features for stop getting better?** Here that thing is digit classification. We truncate the 1797 × 64 digit matrix to rank `k`, fit a plain least-squares classifier on the `k` resulting features, and score it on a held-out third of the data.

The classifier is deliberately the cheapest one available — one-hot targets and `np.linalg.lstsq`, which is itself an SVD solve. No new library, and the whole sweep over every `k` from 1 to 64 runs in under a second.

Two things to look for, and the second is the surprise:

1. **The elbow.** Accuracy climbs steeply through the first dozen or so components and then crawls. The cell prints the smallest `k` landing within one point of full-rank accuracy, and it is typically about a third of the 64 available.
2. **Truncation can beat full rank.** The best accuracy usually sits *below* `k = 64`. Dropping the smallest singular directions removes variance the classifier would otherwise fit, so truncation is acting as regularization — it is not merely a cheaper approximation to the full-rank answer, it is sometimes a better one.

> 🇪🇸 El error de reconstrucción es un sustituto; nadie entrega error de reconstrucción.
>
> Haz la pregunta que decide de verdad un rango: **¿en qué `k` deja de mejorar aquello para lo que construiste las variables?** Aquí es clasificar dígitos: truncamos la matriz de 1797 × 64 a rango `k`, ajustamos un clasificador de mínimos cuadrados sobre las `k` variables resultantes y lo evaluamos sobre un tercio reservado.
>
> El clasificador es el más barato posible — objetivos one-hot y `np.linalg.lstsq`, que es a su vez una resolución por SVD — y el barrido completo de `k = 1` a `64` corre en menos de un segundo.
>
> Dos cosas que mirar, y la segunda es la sorpresa:
>
> 1. **El codo.** La exactitud sube rápido durante la primera docena de componentes y luego se arrastra; la celda imprime el menor `k` que queda a un punto del rango completo, y suele ser un tercio de los 64.
> 2. **Truncar puede superar al rango completo.** La mejor exactitud suele estar *por debajo* de `k = 64`: descartar las direcciones singulares más pequeñas elimina varianza que el clasificador ajustaría, así que la truncación actúa como regularización.

In [ ]:
# A fixed 70/30 split of the real digits, and the PCA basis of the training
# half only — fitting the basis on the test rows would leak. The split gets its
# own generator so that re-running any cell above cannot change it.
split = np.random.default_rng(12).permutation(len(D))
cut = int(0.7 * len(D))
train_idx, test_idx = split[:cut], split[cut:]

D_train, D_test = D[train_idx], D[test_idx]
y_train, y_test = digit_labels[train_idx], digit_labels[test_idx]

pixel_mean = D_train.mean(axis=0)
U_d, S_d, Vt_d = np.linalg.svd(D_train - pixel_mean, full_matrices=False)

targets_train = np.eye(10)[y_train]

def accuracy_at_rank(k):
    """Least-squares classifier on the top-k principal directions."""
    Z_train = np.column_stack([
        np.ones(len(D_train)), (D_train - pixel_mean) @ Vt_d[:k].T])
    Z_test = np.column_stack([
        np.ones(len(D_test)), (D_test - pixel_mean) @ Vt_d[:k].T])
    W, *_ = np.linalg.lstsq(Z_train, targets_train, rcond=None)
    return np.mean((Z_test @ W).argmax(axis=1) == y_test)

rank_grid = np.arange(1, D.shape[1] + 1)
accuracy_curve = np.array([accuracy_at_rank(int(k)) for k in rank_grid])

full_rank_accuracy = accuracy_curve[-1]
best_k = int(rank_grid[np.argmax(accuracy_curve)])
within_one_point = int(rank_grid[
    np.argmax(accuracy_curve >= full_rank_accuracy - 0.01)])

print("Train / Entrenamiento:", D_train.shape,
      "  Test / Prueba:", D_test.shape)
print()
print("Full rank (k = 64) accuracy / Exactitud a rango completo:",
      f"{full_rank_accuracy:.4f}")
print("Best accuracy / Mejor exactitud:",
      f"{accuracy_curve.max():.4f}", f"at k = {best_k}")
print("First k within 1 point of full rank / Primer k a 1 punto:",
      within_one_point,
      f"({100 * within_one_point / D.shape[1]:.0f}% of the features)")

### Interactive downstream-task explorer / Explorador interactivo de la tarea final

Slide the rank and watch the classifier's accuracy against what the same rank does to a single digit image. Low rank blurs the digit into a smear long before accuracy collapses — the classifier does not need the digit to *look* right, only to be *separable*.

That gap is the reason reconstruction error is a bad stopping rule. It measures how well you kept the picture; the metric you ship measures whether the decision survived.

> 🇪🇸 Desliza el rango y compara la exactitud del clasificador con lo que ese mismo rango le hace a un dígito concreto. Un rango bajo convierte el dígito en una mancha mucho antes de que la exactitud se desplome: el clasificador no necesita que el dígito *se vea* bien, solo que sea *separable*.
>
> Esa brecha es la razón por la que el error de reconstrucción es una mala regla de parada: mide lo bien que conservaste la imagen, no si sobrevivió la decisión.

In [ ]:
downstream_rank = widgets.IntSlider(
    value=16,
    min=1,
    max=64,
    step=1,
    description="Rank k / Rango k:",
    continuous_update=False,
    style={"description_width": "130px"},
)

digit_pick = widgets.IntSlider(
    value=0,
    min=0,
    max=19,
    step=1,
    description="Test digit / Dígito:",
    continuous_update=False,
    style={"description_width": "130px"},
)

def explore_downstream(k, which):
    accuracy = accuracy_curve[k - 1]
    reconstructed = (pixel_mean
                     + ((D_test - pixel_mean) @ Vt_d[:k].T) @ Vt_d[:k])
    error = relative_error(D_test, reconstructed)

    print("Rank / Rango:", k, "of / de 64")
    print("Held-out accuracy / Exactitud reservada:", f"{accuracy:.4f}")
    print("Full-rank accuracy / A rango completo:",
          f"{full_rank_accuracy:.4f}",
          f"({accuracy - full_rank_accuracy:+.4f})")
    print("Reconstruction error / Error de reconstrucción:", f"{error:.4f}")

    fig, axes = plt.subplots(1, 3, figsize=(11.0, 3.4))
    axes[0].plot(rank_grid, accuracy_curve, "-", linewidth=1.5)
    axes[0].axvline(k, color="0.6", linewidth=1)
    axes[0].axhline(full_rank_accuracy, color="0.75", linestyle=":",
                    linewidth=1)
    axes[0].set_xlabel("Rank k / Rango k")
    axes[0].set_ylabel("Held-out accuracy / Exactitud")
    axes[0].set_title("Accuracy against rank / Exactitud frente al rango")
    axes[0].grid(alpha=0.25)

    axes[1].imshow(D_test[which].reshape(8, 8), cmap="gray_r")
    axes[1].set_title(f"Original — label / etiqueta {y_test[which]}")
    axes[2].imshow(reconstructed[which].reshape(8, 8), cmap="gray_r")
    axes[2].set_title(f"Rank {k} / Rango {k}")
    for ax in axes[1:]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

downstream_output = widgets.interactive_output(
    explore_downstream,
    {"k": downstream_rank, "which": digit_pick},
)

display(widgets.VBox([downstream_rank, digit_pick, downstream_output]))

## 12.6 NMF — parts you can name / NMF — partes que puedes nombrar

The SVD is optimal, so why would anyone use anything else?

Because optimal-in-Frobenius-norm is not the only thing a factorization can be asked for. The digit matrix `D` is nonnegative — pixel intensities, and a negative amount of ink means nothing. The SVD does not know that. Its components contain negative entries and are only interpretable as *corrections*: component 3 subtracts what components 1 and 2 over-added. You cannot point at one and say what it is.

NMF adds one constraint, `W ≥ 0` and `H ≥ 0`, and gives up optimality to get it. Since every component adds ink and nothing removes it, each one has to be a piece of a digit on its own. The next cell fits both at rank 16 and measures both sides of the trade:

- **Error.** NMF must be worse; Eckart–Young guarantees it. Measure by how much.
- **Sparsity.** Count how many component entries are effectively zero. That number is what "parts-based" means quantitatively, and it is where NMF wins by a wide margin.

> 🇪🇸 Si la SVD es óptima, ¿por qué usar otra cosa?
>
> Porque óptima-en-norma-de-Frobenius no es lo único que se le puede pedir a una factorización. La matriz de dígitos `D` es no negativa — son intensidades, y una cantidad negativa de tinta no significa nada. La SVD no lo sabe: sus componentes tienen entradas negativas y solo se interpretan como *correcciones*, donde la componente 3 resta lo que las componentes 1 y 2 añadieron de más. No puedes señalar una y decir qué es.
>
> NMF añade una restricción, `W ≥ 0` y `H ≥ 0`, y renuncia a la optimalidad para conseguirla. Como cada componente añade tinta y ninguna la quita, cada una tiene que ser por sí sola un trozo de dígito.
>
> La celda siguiente ajusta ambas a rango 16 y mide los dos lados del intercambio: el **error**, que en NMF debe ser peor (Eckart–Young lo garantiza), y la **dispersión**, que es lo que "basado en partes" significa cuantitativamente.

In [ ]:
k_parts = 16

nmf = NMF(n_components=k_parts, init="nndsvda", max_iter=2000,
          tol=1e-5, random_state=0)
W = nmf.fit_transform(D)
H = nmf.components_

U_D, S_D, Vt_D = np.linalg.svd(D, full_matrices=False)
D_svd = (U_D[:, :k_parts] * S_D[:k_parts]) @ Vt_D[:k_parts]

near_zero = lambda M: np.mean(np.abs(M) < 0.01 * np.abs(M).max())

print(f"Rank / Rango {k_parts} on / sobre {D.shape}")
print()
print("Relative error / Error relativo:")
print("   SVD (optimal / óptima):", f"{relative_error(D, D_svd):.4f}")
print("   NMF                   :", f"{relative_error(D, W @ H):.4f}")
print()
print("Component entries that are effectively zero / Entradas casi nulas:")
print("   SVD components / componentes:", f"{100 * near_zero(Vt_D[:k_parts]):.1f}%")
print("   NMF components / componentes:", f"{100 * near_zero(H):.1f}%")
print("   Most negative SVD entry / Entrada más negativa:",
      f"{Vt_D[:k_parts].min():.3f}")
print("   Most negative NMF entry / Entrada más negativa:", f"{H.min():.3f}")

fig, axes = plt.subplots(2, k_parts // 2, figsize=(12.0, 3.6))
for i, ax in enumerate(axes.ravel()):
    ax.imshow(H[i].reshape(8, 8), cmap="gray_r")
    ax.axis("off")
fig.suptitle("NMF components — each one is a stroke, not a correction "
             "/ cada componente es un trazo, no una corrección")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(2, k_parts // 2, figsize=(12.0, 3.6))
for i, ax in enumerate(axes.ravel()):
    ax.imshow(Vt_D[i].reshape(8, 8), cmap="coolwarm",
              vmin=-np.abs(Vt_D[i]).max(), vmax=np.abs(Vt_D[i]).max())
    ax.axis("off")
fig.suptitle("SVD components — red adds, blue subtracts "
             "/ el rojo suma, el azul resta")
plt.tight_layout()
plt.show()

print()
print("EN: the SVD wins on error by a few points and loses on readability")
print("    completely. Which one you want depends on whether a human has to")
print("    explain the components afterwards.")
print("ES: la SVD gana en error por unos puntos y pierde por completo en")
print("    legibilidad. Cuál quieres depende de si después alguien tiene que")
print("    explicar las componentes.")

## 12.7 Sizing your own problem / Dimensiona tu propio problema

Everything so far has been on matrices this workshop hands you. The last tool is for the matrix you will bring tomorrow.

Set `m`, `n` and `k`, and read back, for every method: the predicted flops from §12.2's table, the bytes the factors occupy, the compression against storing `A` outright, and a wall-clock estimate.

The estimate is calibrated on *this* machine, right now. The cell times one QR factorization of known flop count, divides to get an effective rate in GFLOP/s, and multiplies every other formula through it.

**Read it as an order of magnitude, not a benchmark.** Measured against the sweep in §12.3, this estimator lands within a factor of about two for QR and SVD, and over-predicts LU by up to six times, because LAPACK's `getrf` is exceptionally well optimized and the `9n³` eigendecomposition constant is conservative. It answers *"seconds, minutes, or overnight?"* — which is the question you actually have before you start a job — and it does not answer anything finer than that.

> 🇪🇸 Todo lo anterior ha sido sobre matrices que el taller te da. La última herramienta es para la matriz que traerás mañana.
>
> Fija `m`, `n` y `k`, y lee para cada método: los flops predichos por la tabla de §12.2, los bytes que ocupan los factores, la compresión frente a guardar `A` entera, y una estimación de tiempo de reloj.
>
> La estimación se calibra en *esta* máquina y ahora: la celda cronometra una QR de flops conocidos, divide para obtener una tasa efectiva en GFLOP/s y pasa por ella el resto de fórmulas.
>
> **Léelo como orden de magnitud, no como benchmark.** Frente al barrido de §12.3, este estimador acierta dentro de un factor de dos para QR y SVD, y sobreestima LU hasta seis veces, porque `getrf` de LAPACK está excepcionalmente optimizado y la constante `9n³` es conservadora. Responde a *"¿segundos, minutos o toda la noche?"*, que es la pregunta que de verdad tienes antes de lanzar un trabajo.

In [ ]:
# Calibrate on one QR of known flop count, on this machine, right now.
calibration_n = 512
calibration_matrix = np.random.default_rng(2).standard_normal(
    (calibration_n, calibration_n))
calibration_time = best_time(lambda: np.linalg.qr(calibration_matrix))
calibration_flops = FLOPS["QR"](calibration_n, calibration_n, 0)
flop_rate = calibration_flops / calibration_time

print(f"Calibration / Calibración: QR {calibration_n}x{calibration_n} in "
      f"{calibration_time * 1e3:.1f} ms")
print(f"Effective rate / Tasa efectiva: {flop_rate / 1e9:.1f} GFLOP/s")
print()

rows_slider = widgets.IntSlider(
    value=10_000, min=100, max=200_000, step=100,
    description="Rows m / Filas m:", continuous_update=False,
    style={"description_width": "150px"}, readout_format=",d")
cols_slider = widgets.IntSlider(
    value=500, min=10, max=5_000, step=10,
    description="Columns n / Columnas n:", continuous_update=False,
    style={"description_width": "150px"}, readout_format=",d")
target_rank = widgets.IntSlider(
    value=20, min=1, max=500, step=1,
    description="Target rank k / Rango k:", continuous_update=False,
    style={"description_width": "150px"})

def human_time(seconds):
    if seconds < 1e-3:
        return f"{seconds * 1e6:.0f} us"
    if seconds < 1:
        return f"{seconds * 1e3:.0f} ms"
    if seconds < 90:
        return f"{seconds:.1f} s"
    if seconds < 5400:
        return f"{seconds / 60:.1f} min"
    return f"{seconds / 3600:.1f} h"

def human_bytes(n_bytes):
    for unit in ("B", "KB", "MB", "GB", "TB"):
        if n_bytes < 1024 or unit == "TB":
            return f"{n_bytes:,.1f} {unit}"
        n_bytes /= 1024

def budget(m, n, k):
    if n > m:
        m, n = n, m
        print("EN: m and n swapped so that m >= n, as the formulas assume.")
        print("ES: se intercambian m y n para que m >= n, como suponen las")
        print("    fórmulas.")
        print()

    k = min(k, n)
    raw_bytes = m * n * 8

    # Numbers each method has to keep, in float64.
    STORED = {
        "Cholesky": n * (n + 1) / 2,
        "LU": n * n,
        "QR": m * n + n * (n + 1) / 2,
        "Eigendecomposition": n * n + n,
        "Thin SVD": m * n + n + n * n,
        "Randomized SVD": k * (m + n + 1),
    }

    # Cholesky, LU and the eigendecomposition are only defined for a square
    # matrix, so their formulas must not be quoted for a rectangular one.
    SQUARE_ONLY = {"Cholesky", "LU", "Eigendecomposition"}
    square = m == n

    print(f"A is {m:,} x {n:,} float64 = {human_bytes(raw_bytes)},"
          f" target rank k = {k}")
    if not square:
        print("A is rectangular / A es rectangular — Cholesky, LU and the")
        print("eigendecomposition do not apply / no se aplican.")
    print()
    print("method / método".ljust(20)
          + "flops".rjust(11) + "est. time".rjust(11)
          + "factors".rjust(13) + "  vs A")
    print("-" * 68)
    for name, formula in FLOPS.items():
        if name in SQUARE_ONLY and not square:
            print(name.ljust(20)
                  + "n/a — needs a square matrix / requiere una cuadrada".rjust(
                      41))
            continue
        flops = formula(m, n, k)
        stored_bytes = STORED[name] * 8
        print(name.ljust(20)
              + f"{flops:10.2e}"
              + human_time(flops / flop_rate).rjust(11)
              + human_bytes(stored_bytes).rjust(13)
              + f"  {stored_bytes / raw_bytes:6.2f}x")

    print()
    full = FLOPS["Thin SVD"](m, n, k)
    randomized = FLOPS["Randomized SVD"](m, n, k)
    print(f"Full SVD / randomized SVD at k = {k}: "
          f"{full / randomized:.1f}x more work / más trabajo")
    print("EN: that ratio is the whole argument for not computing what you")
    print("    intend to throw away.")
    print("ES: esa razón es todo el argumento para no calcular lo que vas a")
    print("    tirar.")

budget_output = widgets.interactive_output(
    budget,
    {"m": rows_slider, "n": cols_slider, "k": target_rank},
)

display(widgets.VBox([rows_slider, cols_slider, target_rank, budget_output]))

## Quick decision challenge / Reto rápido de decisión

Six situations, one factorization each. Pick before you read the answer.

> 🇪🇸 Seis situaciones, una factorización para cada una. Elige antes de leer la respuesta.

In [ ]:
scenario = widgets.Dropdown(
    options=[
        ("A 50,000 x 300 design matrix, fit once / una vez", "tall"),
        ("A 4,000 x 4,000 covariance matrix, 500 right-hand sides",
         "spd_many"),
        ("A 200,000 x 50,000 sparse ratings matrix, top 30 components",
         "sparse_lowrank"),
        ("A 3,000 x 3,000 gene-expression matrix, components must be "
         "explainable", "nonneg"),
        ("A Markov transition matrix — what is the steady state?", "markov"),
        ("A 40,000 x 8,000 dense matrix, top 100 components", "dense_lowrank"),
    ],
    value="tall",
    description="Situation / Situación:",
    style={"description_width": "150px"},
    layout=widgets.Layout(width="720px"),
)

ANSWERS = {
    "tall": (
        "QR (np.linalg.lstsq)",
        "Tall and fit once: this is least squares. Never form X^T X — QR "
        "solves it at kappa(X) instead of kappa(X)^2.",
        "Alta y ajustada una vez: esto es mínimos cuadrados. Nunca formes "
        "X^T X — QR lo resuelve con kappa(X) en vez de kappa(X)^2.",
    ),
    "spd_many": (
        "Cholesky (cho_factor once, then cho_solve 500 times)",
        "A covariance matrix is symmetric positive definite, so Cholesky is "
        "half the flops of LU. 500 right-hand sides make the factor-once "
        "saving O(n^3 + m n^2) against O(m n^3).",
        "Una matriz de covarianza es simétrica definida positiva, así que "
        "Cholesky cuesta la mitad que LU. Con 500 lados derechos, factorizar "
        "una vez es O(n^3 + m n^2) frente a O(m n^3).",
    ),
    "sparse_lowrank": (
        "Lanczos / truncated SVD (scipy.sparse.linalg.svds)",
        "Sparse and only 30 components wanted. A dense SVD would first "
        "densify a 10-billion-entry matrix to compute 30 vectors. Lanczos "
        "costs O(k * nnz(A)) and never densifies.",
        "Dispersa y solo se quieren 30 componentes. Una SVD densa primero "
        "densificaría una matriz de 10 mil millones de entradas para calcular "
        "30 vectores. Lanczos cuesta O(k * nnz(A)) y nunca densifica.",
    ),
    "nonneg": (
        "NMF",
        "Expression counts are nonnegative and the components have to be "
        "readable. This is the one case where giving up Eckart-Young "
        "optimality is the right call — you are buying interpretability with "
        "error.",
        "Los conteos de expresión son no negativos y las componentes deben ser "
        "legibles. Es el único caso donde renunciar a la optimalidad de "
        "Eckart-Young es correcto: compras interpretabilidad con error.",
    ),
    "markov": (
        "Eigendecomposition (or power iteration)",
        "Steady state is the eigenvector for eigenvalue 1 — the question "
        "'what does repeated application converge to?' from section 08. If "
        "you only need the top one, power iteration beats a full "
        "eigendecomposition.",
        "El estado estacionario es el autovector del autovalor 1: la pregunta "
        "'¿a qué converge la aplicación repetida?' de la sección 08. Si solo "
        "necesitas el primero, la iteración de potencias gana a una "
        "descomposición completa.",
    ),
    "dense_lowrank": (
        "Randomized SVD (sklearn.utils.extmath.randomized_svd)",
        "Dense, so Lanczos has no sparsity to exploit, but 100 components out "
        "of 8,000 means a full SVD does eighty times the necessary work. "
        "Randomized SVD costs O(mnk).",
        "Densa, así que Lanczos no tiene dispersión que aprovechar, pero 100 "
        "componentes de 8.000 significa que una SVD completa hace ochenta "
        "veces el trabajo necesario. La SVD aleatorizada cuesta O(mnk).",
    ),
}

def answer(key):
    pick, en, es = ANSWERS[key]
    print("Use / Usa:", pick)
    print()
    print("EN:", en)
    print()
    print("ES:", es)

challenge_output = widgets.interactive_output(answer, {"key": scenario})

display(widgets.VBox([scenario, challenge_output]))

## What just happened / Qué acaba de pasar

You started with six factorizations that looked like six unrelated algorithms and ended with a decision procedure.

**They are all constrained optimizations.** QR minimizes `‖y − Xβ‖` subject to an orthonormal `Q`; the truncated SVD minimizes `‖A − B‖_F` subject to `rank(B) ≤ k`; NMF minimizes the same thing subject to nonnegativity; Cholesky and LU optimize nothing at all and are pure setup for cheap solves. The constraint is what gives each factorization its shape, and its shape is what makes one question easy.

**Cost has two halves, and only one of them is in the big-O.** The flop table told you a full SVD is about `39×` a Cholesky, and you measured a ratio near that. It also told you every dense factorization is cubic, and you measured slopes near 2.4 — because parallelism and cache reuse improve as `n` grows. The exponent is the theory; the ratio is the part that survives contact with a real machine.

**The most expensive mistake in this notebook cost one line.** Writing `solve(X.T @ X, X.T @ y)` instead of a QR squares the condition number, and on an ordinary degree-10 polynomial fit to real airline data that turned a coefficient error of `2e-14` into `2e-03`. The residuals were identical to six decimal places the whole time, so the check most people would run says nothing is wrong.

**Optimal is not the same as useful.** The SVD is provably the best rank-`k` approximation, and it is still the wrong choice when you need components a human can name, or when the matrix is sparse and you want thirty of eight thousand components. NMF is worse on error by about four points and has over 80% of its component entries at zero against the SVD's 21%; that sparsity is the product, not a defect.

**Rank should be chosen against the metric you ship.** Reconstruction error said one thing about the digits; held-out accuracy said something better — that truncation below full rank was not just cheaper but genuinely *more accurate*, because dropping the smallest singular directions regularized the classifier.

### Where this goes next / Dónde continúa esto

- `scipy.linalg` — `lu_factor`/`lu_solve`, `cho_factor`/`cho_solve`, `qr`, `svd` with `lapack_driver` options.
- `scipy.sparse.linalg.svds` and `sklearn.utils.extmath.randomized_svd` — the two ways to not compute what you will throw away.
- **Deep dive 13** — the same question one order up. Tensor rank is not the tidy generalization of matrix rank it appears to be, and Eckart–Young has no direct analogue there.

> 🇪🇸 Empezaste con seis factorizaciones que parecían seis algoritmos sin relación y terminaste con un procedimiento de decisión.
>
> **Todas son optimizaciones con restricciones**, y la restricción es lo que da forma a los factores; esa forma es lo que vuelve fácil una pregunta.
>
> **El coste tiene dos mitades y solo una está en la big-O.** La tabla predijo que una SVD completa cuesta unas `39×` una Cholesky y mediste algo cercano; también dijo que todo es cúbico y mediste pendientes cerca de 2.4, porque el paralelismo y la caché mejoran al crecer `n`. El exponente es la teoría; la razón es lo que sobrevive al contacto con una máquina real.
>
> **El error más caro de este cuaderno costaba una línea.** Escribir `solve(X.T @ X, X.T @ y)` en vez de una QR eleva al cuadrado el número de condición, y sobre un ajuste polinómico normal a datos reales convirtió un error de `2e-14` en uno de `2e-03` — con residuos idénticos en seis decimales todo el tiempo.
>
> **Óptimo no es lo mismo que útil.** La SVD es demostrablemente la mejor aproximación de rango `k`, y sigue siendo la elección equivocada cuando necesitas componentes que alguien pueda nombrar.
>
> **El rango se elige contra la métrica que entregas.** El error de reconstrucción decía una cosa sobre los dígitos; la exactitud reservada decía algo mejor: truncar por debajo del rango completo no solo era más barato sino genuinamente *más preciso*.

---

## Done with this deep dive / Fin de este estudio a fondo

Next deep dive / Siguiente estudio a fondo: **13 · Tensor factorizations / Factorizaciones tensoriales** — [open in Colab](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/13-tensor-factorizations.ipynb).

[← Workshop site / Sitio del taller](https://project-delphi.github.io/tensors-workshop/) · [All notebooks / Todos los notebooks](https://project-delphi.github.io/tensors-workshop/notebooks.html) · [Handbook / Manual](https://project-delphi.github.io/tensors-workshop/tensors_workshop_plan_with_quizzes.html)